# Lecture: Clinical Research Informatics - Sommer Semester 2026
Exercise Sheet: 4   

Additional information:   
used additional packages: none   
dataset used: Dyan186_Purdy2_61b11ebf-b7c1-0bb8-2ded-c5405bd62015.json   

In [ ]:
# importing dependencies
import json
import pandas as pd

from datetime import date, datetime

In [3]:
# loading the file from data folder
patient_file = "data/Dyan186_Purdy2_61b11ebf-b7c1-0bb8-2ded-c5405bd62015.json"

with open(patient_file, "r") as f:
    data = json.load(f)

In [ ]:
# inspecting raw structure

# top-level keys
print(data.keys())
# number of entries
print(len(data["entry"]))

dict_keys(['resourceType', 'type', 'entry'])
584


In [ ]:
# showing all unique top-level resources
resource_types = {entry["resource"]["resourceType"] for entry in data["entry"]}

for rt in sorted(resource_types):
    print(f"{rt:<25} {len(rt)}")

# some counts are part of a url

CarePlan                  8
CareTeam                  8
Claim                     5
Condition                 9
Device                    6
DiagnosticReport          16
DocumentReference         17
Encounter                 9
ExplanationOfBenefit      20
Immunization              12
MedicationRequest         17
Observation               11
Patient                   7
Procedure                 9
Provenance                10


In [ ]:
# 1.2. inspecting raw structure of patient resourceType
patient_info = [entry["resource"] for entry in data["entry"] 
           if entry["resource"]["resourceType"] == "Patient"]
patient = patient_info[0]

print(json.dumps(patient, indent=2))

{
  "resourceType": "Patient",
  "id": "61b11ebf-b7c1-0bb8-2ded-c5405bd62015",
  "meta": {
    "profile": [
      "http://hl7.org/fhir/us/core/StructureDefinition/us-core-patient"
    ]
  },
  "text": {
    "status": "generated",
    "div": "<div xmlns=\"http://www.w3.org/1999/xhtml\">Generated by <a href=\"https://github.com/synthetichealth/synthea\">Synthea</a>.Version identifier: 2e587be\n .   Person seed: 6757553199843334228  Population seed: 1712256522158</div>"
  },
  "extension": [
    {
      "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-race",
      "extension": [
        {
          "url": "ombCategory",
          "valueCoding": {
            "system": "urn:oid:2.16.840.1.113883.6.238",
            "code": "2106-3",
            "display": "White"
          }
        },
        {
          "url": "text",
          "valueString": "White"
        }
      ]
    },
    {
      "url": "http://hl7.org/fhir/us/core/StructureDefinition/us-core-ethnicity",
      "exte

In [28]:
# inspecting available keys
patient_keys = {key for resource in patient_info for key in resource.keys()}
print(patient_keys)

{'address', 'identifier', 'telecom', 'birthDate', 'communication', 'extension', 'gender', 'maritalStatus', 'multipleBirthBoolean', 'meta', 'name', 'resourceType', 'id', 'text'}


In [ ]:
# creating a function to extract patient information
def extract_patient_info(patient):
    name = patient.get("name", [{}])[0]
    full_name = f"{' '.join(name.get('given', []))} {name.get('family', 'N/A')}".strip()
    
    address = patient.get("address", [{}])[0]

    print(f"Full Name:       {full_name}")
    print(f"Date of Birth:   {patient.get('birthDate', 'N/A')}")
    print(f"Country:         {address.get('country', 'N/A')}")
    print(f"Gender:          {patient.get('gender', 'N/A')}")
    print(f"Marital Status:  {patient.get('maritalStatus', {}).get('text', 'N/A')}")
    print(f"Multiple Birth?: {patient.get('multipleBirthBoolean', 'N/A')}")

# calling it
extract_patient_info(patient)

Full Name:       Dyan186 Joey457 Purdy2
Date of Birth:   2013-11-01
Country:         US
Gender:          female
Marital Status:  Never Married
Multiple Birth?: False


In [47]:
# 2.1 conditions' information
conditions = [entry["resource"] for entry in data["entry"]
              if entry["resource"]["resourceType"] == "Condition"]

print(f"Total conditions found: {len(conditions)}\n")
print(f"{'#':<4} {'Onset Date':<14} {'Diagnosis'}")
print("-" * 55)

for i, cond in enumerate(conditions, 1):
    diagnosis = cond.get("code", {}).get("coding", [{}])[0].get("display", "Unknown")
    onset = cond.get("onsetDateTime", cond.get("onsetPeriod", {}).get("start", "Unknown"))
    onset_date = onset[:10] if onset != "Unknown" else "Unknown"
    print(f"{i:<4} {onset_date:<14} {diagnosis}")

Total conditions found: 20

#    Onset Date     Diagnosis
-------------------------------------------------------
1    2014-04-11     Medication review due (situation)
2    2014-09-12     Gunshot wound (disorder)
3    2014-09-12     Bullet wound
4    2014-10-10     Medication review due (situation)
5    2015-04-10     Medication review due (situation)
6    2015-04-29     Streptococcal sore throat (disorder)
7    2015-10-09     Medication review due (situation)
8    2017-07-19     Childhood asthma
9    2017-09-08     Medication review due (situation)
10   2017-09-22     Streptococcal sore throat (disorder)
11   2018-05-09     Otitis media
12   2018-07-29     Sprain (morphologic abnormality)
13   2018-07-29     Sprain of wrist
14   2018-10-12     Medication review due (situation)
15   2018-12-21     Viral sinusitis (disorder)
16   2019-04-10     Otitis media
17   2019-10-31     Homeless (finding)
18   2020-12-24     Viral sinusitis (disorder)
19   2021-01-01     Medication review due (si

In [48]:
# 2.2 age at the time of the first asthma diagnosis
asthma_conditions = [
    c for c in conditions
    if "asthma" in c.get("code", {}).get("coding", [{}])[0].get("display", "").lower()
]

# sort by onset date
asthma_conditions.sort(key=lambda c: c.get("onsetDateTime",
                        c.get("onsetPeriod", {}).get("start", ""))[:10])

first_asthma = asthma_conditions[0]
diagnosis_name = first_asthma.get("code", {}).get("coding", [{}])[0].get("display", "Unknown")
onset_str = first_asthma.get("onsetDateTime",
            first_asthma.get("onsetPeriod", {}).get("start", ""))[:10]

dob_dt    = datetime.strptime(birth_date, "%Y-%m-%d").date()
onset_dt  = datetime.strptime(onset_str, "%Y-%m-%d").date()
# first asthma diagnosis age
age_years = (onset_dt - dob_dt).days // 365

print(f"Diagnosis     : {diagnosis_name}")
print(f"Date of Birth : {birth_date}")
print(f"Onset Date    : {onset_str}")
print(f"Age at first asthma diagnosis: {age_years} years old")

Diagnosis     : Childhood asthma
Date of Birth : 2013-11-01
Onset Date    : 2017-07-19
Age at first asthma diagnosis: 3 years old


In [49]:
# 3.1 make a pandas DataFrame
observations = [entry["resource"] for entry in data["entry"]
                if entry["resource"]["resourceType"] == "Observation"]

rows = []
for obs in observations:
    # correct value types
    value, unit = "", ""
    if "valueQuantity" in obs:
        value = obs["valueQuantity"].get("value", "")
        unit  = obs["valueQuantity"].get("unit", "")
    elif "valueCodeableConcept" in obs:
        value = obs["valueCodeableConcept"].get("coding", [{}])[0].get("display", "")
    elif "valueString" in obs:
        value = obs["valueString"]

    rows.append({
        "id":          obs.get("id", ""),
        "date":        obs.get("effectiveDateTime", "")[:10],
        "observation": obs.get("code", {}).get("coding", [{}])[0].get("display", ""),
        "value":       value,
        "unit":        unit,
        "status":      obs.get("status", ""),
    })

df_observations = pd.DataFrame(rows)

print(f"Total observations: {len(df_observations)}\n")
# printing the first few rows of the DataFrame
df_observations.head()

Total observations: 229



,id,date,observation,value,unit,status
0,a2ef7c1a-e7dc-30a6-2f91-8f580a61b645,2014-04-11,Body Height,68,cm,final
1,85e07fdc-f407-f489-cd32-a40a35289660,2014-04-11,Pain severity - 0-10 verbal numeric rating [Sc...,3,{score},final
2,543cc531-e150-c779-3270-2907b0596eb6,2014-04-11,Body Weight,6.9,kg,final
3,cd8e1861-8ad2-f004-b8bf-ef6d65540103,2014-04-11,Weight-for-length Per age and sex,3.9956,%,final
4,b1a3f5f5-7ab9-5937-9f70-c6fe8722c763,2014-04-11,Head Occipital-frontal circumference Percentile,94.06,%,final
